<!-- cabecera-entorno -->
## Antes de empezar

**Clase 8 · Principios de visualización** — Bloque 3 · Reto. Este cuaderno lo recorre **usted solo**,
leyendo: cada tarea trae la explicación y los comandos que necesita. El profesor circula por el salón
resolviendo dudas. Es el entregable de la clase.

**La rutina de siempre:** `git pull` antes de clase, y el entorno virtual activo (`(.venv)` en la
terminal). Si va a modificar este archivo, trabaje sobre una copia: duplique `reto.ipynb` como
`reto_mio.ipynb` y edite el duplicado. Así `git pull` nunca le reclama.

**Si la celda de abajo falla, no siga:** la respuesta está en el manual del entorno,
[`../INSTALACION.md`](../INSTALACION.md).

| Si ve esto | Qué pasó | Dónde se arregla |
|------------|----------|------------------|
| `ModuleNotFoundError` | El entorno virtual no está activo, o VSCode eligió otro intérprete | Manual, secciones 6.3 y 8.4, y problema 5 |
| `FileNotFoundError` al leer el CSV | El notebook se abrió desde otra carpeta, o falta hacer `git pull` | Manual, problema 6 |
| El kernel no aparece en VSCode | Falta la extensión Jupyter o `ipykernel` dentro del entorno | Manual, problema 4 |

In [ ]:
# Verificación del entorno. Si algo falla aquí, la solución está en ../INSTALACION.md
import sys
from pathlib import Path

try:
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns
except ModuleNotFoundError as error:
    raise ModuleNotFoundError(
        f"Falta la librería '{error.name}'. Active el entorno virtual y seleccione el intérprete "
        ".venv en VSCode (Ctrl+Shift+P > Python: Select Interpreter), luego reinicie el kernel. "
        "Ver ../INSTALACION.md, problema 5."
    ) from error

print("Intérprete:", sys.executable)
if ".venv" not in sys.executable:
    print("AVISO: este no parece el Python del entorno virtual. En VSCode: Ctrl+Shift+P >",
          "'Python: Select Interpreter' > el que dice .venv, y reinicie el kernel.")

RUTA_VERIFICACION = "../datos/EJECUCION_PRESUPUESTAL.csv"
if Path(RUTA_VERIFICACION).exists():
    print("Datos: encontrados en", RUTA_VERIFICACION)
else:
    print("FALTA el archivo", RUTA_VERIFICACION, "- abra en VSCode la carpeta raíz del curso",
          "y ejecute 'git pull'. Ver ../INSTALACION.md, problema 6.")

# Clase 8 · Reto — Cinco figuras publication-ready sobre ejecución presupuestal

**Equipo:**

**Fecha:**

**Dataset:** `../datos/EJECUCION_PRESUPUESTAL.csv` — ejecución del Presupuesto General de la
Nación, vía datos.gov.co.
**Consigna completa:** `reto.md`

3.645 filas x 21 columnas. Una fila = un rubro presupuestal de una entidad, con su fuente de
financiación.

## Sí, es el mismo dataset del demo. Es a propósito

En el resto del curso el reto usa datos que no vio en el demo. **Esta clase es la excepción
declarada.**

Hoy lo que se evalúa no es su capacidad de cargar y entender datos nuevos: es su **criterio visual**.
Si tuviera que aprenderse un dataset nuevo, gastaría 20 de sus 60 minutos en limpieza y llegaría a
los gráficos con 40.

**Lo que cambia no son los datos: son los gráficos.** Cuatro de las cinco figuras son de tipos que no
aparecieron en el demo: barras agrupadas, dispersión, apiladas al 100% y small multiples.

> Los mismos números, cinco preguntas distintas, cinco gráficos distintos. Si el gráfico correcto
> dependiera de los datos, esto sería trivial. Depende de la pregunta.

### Las columnas que va a usar

| Columna | Qué es |
|---------|--------|
| `Nombre Sector` | 32 sectores. El nivel grueso |
| `Nombre Entidad` | 221 entidades. El nivel fino |
| `Nombre Nivel Uno Rubro` | Tres categorías: funcionamiento, inversión y servicio de la deuda |
| `Apropiación Vigente` | Lo que le autorizaron gastar. El techo |
| `Compromisos` | Lo que ya comprometió en contratos firmados |
| `Obligaciones` | Lo que ya recibió y debe pagar |
| `Pagos` | Lo que efectivamente salió |

El **porcentaje de ejecución** es `Compromisos / Apropiación Vigente * 100`.

## Cómo se recorre este cuaderno

Usted trabaja solo. Nadie va a dictar los pasos desde el tablero, así que cada tarea trae todo lo que
necesita para resolverse leyendo:

| Parte de la tarea | Qué contiene |
|-------------------|--------------|
| **La pregunta** | Lo que hay que responder, escrito en español |
| **El concepto** | Qué técnica aplica y por qué esa y no otra |
| **Los comandos** | Las instrucciones exactas que va a usar, escritas de forma genérica |
| **Lo que decide usted** | Qué columna, qué recorte, qué color, qué título. Ahí no hay respuesta escrita |
| **La celda de código** | Los pasos numerados en comentarios. Usted escribe las líneas |
| **La comprobación** | `comprobar('TN', ...)` le dice si el resultado es el correcto, sin mostrárselo |

**Las diez tareas (T1 a T10)** están repartidas en tres partes:

| Parte | Tareas | De qué va |
|-------|--------|-----------|
| **A · Los datos** | T1, T2 | Las dos tablas agregadas de las que salen las cinco figuras |
| **B · Las cinco figuras** | T3 a T8 | Una tarea por figura, más la tabla normalizada de la figura 4 |
| **C · Sin el orden** | T9, T10 | El ensamblaje y la exportación. **La que más pesa** |

**Por qué esto sigue siendo un reto y no una copia.** Los comandos se dan; las decisiones no. Usted
elige el recorte, el color, el orden, la unidad del eje y —sobre todo— **el título, que tiene que
decir una conclusión y ser verdadero**. Eso es lo que se evalúa. Las celdas `comprobar(...)` comparan
una huella digital de su resultado con la esperada: nunca revelan la respuesta.

**Las tareas de gráfico se comprueban distinto.** No hay una única figura correcta, así que lo que se
revisa es lo verificable: que esté dibujada, titulada, con los ejes etiquetados, con el eje en cero
donde la regla lo exige, con la línea de referencia donde hace falta y con la misma escala en todos
los paneles. Los puntos 1, 7 y 8 de la checklist —los cinco segundos, la escala de grises y que el
título sea **verdadero**— no los puede revisar ningún programa. Esos los juzga usted, y son los que
más pesan al calificar.

**El reparto del tiempo:** unos 9 minutos por figura. Si se demora 20 en la primera, no termina. La
figura 1 es la del demo: hágala rápido, es el calentamiento.

---

## Paso 0 · Preparación

Las tres celdas de abajo ya están escritas: las librerías, la carga y la limpieza mínima. Ejecútelas
y **lea lo que imprimen**.

**Qué se limpia y por qué.** Solo dos cosas, que son las que impiden graficar bien:

1. El **espacio no separable** (`\xa0`), que parte una categoría en dos que se ven idénticas.
   `str.strip()` no lo arregla, porque no está en los extremos.
2. Las **tres filas con apropiación en cero**, que al calcular el porcentaje de ejecución dividen
   por cero y producen `NaN` o `inf` sin lanzar ningún error.

`Nombre Sector` y `Nombre Entidad` ya están limpios. No gaste tiempo ahí.

In [ ]:
import os
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 25)
pd.set_option('display.width', 200)

# La carpeta donde van a quedar los PNG exportados
os.makedirs('figuras', exist_ok=True)

print('pandas', pd.__version__, '| matplotlib', plt.matplotlib.__version__)

In [ ]:
# Este cuaderno vive en clase08/reto/, y el CSV dos carpetas más arriba, en datasets/
df = pd.read_csv('../datos/EJECUCION_PRESUPUESTAL.csv')

df_limpio = df.copy()

# 1. El espacio no separable, en todas las columnas de texto
for col in df_limpio.columns:
    if pd.api.types.is_string_dtype(df_limpio[col]):
        df_limpio[col] = df_limpio[col].str.replace('\xa0', ' ', regex=False).str.strip()

# 2. Las filas sin apropiación. Se filtran y se dice cuántas: nunca en silencio.
excluidas = (df_limpio['Apropiación Vigente'] == 0).sum()
df_limpio = df_limpio[df_limpio['Apropiación Vigente'] > 0].copy()

print(f'Filas cargadas: {len(df)}')
print(f'Filas excluidas por apropiación en cero: {excluidas}')
print(f'Filas de trabajo: {len(df_limpio)}')
print(f'Sectores: {df_limpio["Nombre Sector"].nunique()} | '
      f'Entidades: {df_limpio["Nombre Entidad"].nunique()}')

# La referencia nacional: se usa en las figuras 3 y 5, y en la tarea 9.
# Es la suma sobre la suma, no el promedio de los porcentajes: responde
# "cuanto del presupuesto del pais esta comprometido", que es otra pregunta.
ejecucion_nacional = (df_limpio['Compromisos'].sum() /
                      df_limpio['Apropiación Vigente'].sum() * 100)
print(f'Ejecución nacional: {ejecucion_nacional:.1f}%')

### El verificador

La celda de abajo define `comprobar(...)` y `comprobar_grafico(...)`. Ejecútela una vez y siga
adelante: es andamiaje del curso, no materia de la clase.

In [ ]:
# Verificador de las diez tareas. Ejecute esta celda una vez y siga adelante.
# No hace falta entenderla hoy: es andamiaje del curso, no materia de la clase.
import hashlib

_RESULTADOS = {}

_CLAVES = ["T1", "T2", "T3", "T4", "T5", "T6", "T7", "T8", "T9", "T10"]

_PISTAS = {
    "T1": "Es un groupby sobre 'Nombre Sector' con las cuatro columnas de plata entre corchetes y .sum() al final, sobre df_limpio. Despues se agregan '% Ejecucion' y 'Apropiacion (billones)' con esos nombres exactos, y se ordena de mayor a menor apropiacion. Si le sobran filas, agrupo sobre df y no sobre df_limpio.",
    "T2": "El mismo groupby, cambiando la columna por la que agrupa a 'Nombre Entidad'. Deben quedar 221 entidades. Las columnas derivadas se calculan igual que en la tarea 1.",
    "T3": "fig1, eje1 = plt.subplots(...) y despues eje1.barh(...). Falta el titulo, alguna etiqueta de eje con su unidad, el eje x arrancando en cero (set_xlim(0, ...)) o las etiquetas directas de valor con eje1.text().",
    "T4": "Las dos series se dibujan con dos llamadas a barh sobre posiciones desplazadas: posiciones - ancho/2 y posiciones + ancho/2. El eje x arranca en cero, y hacen falta el titulo, las dos etiquetas de eje y el porcentaje etiquetado de cada par.",
    "T5": "Guarde el eje: fig3, eje3 = plt.subplots(...). Hace falta la linea de referencia (eje3.axhline(ejecucion_nacional, ...)) y al menos dos entidades anotadas con eje3.annotate(...), ademas del titulo y las dos etiquetas de eje.",
    "T6": "Es una division de cada fila por su propio total: composicion.div(composicion.sum(axis=1), axis=0) * 100. El axis=1 de la suma dice 'sume a lo ancho'; el axis=0 de la division dice 'divida cada fila por su valor'. Cada fila debe sumar 100.",
    "T7": "Las apiladas se arman acumulando: se lleva un vector 'acumulado' y cada capa se dibuja con left=acumulado, sumandole despues los valores de esa capa. El eje x va de 0 a 100, y hacen falta el titulo y las dos etiquetas de eje.",
    "T8": "fig5, ejes5 = plt.subplots(2, 4, figsize=..., sharex=True). Guarde el arreglo completo en ejes5, no un solo panel. Cada panel necesita su titulo. Si el verificador se queja de las escalas, falto sharex=True.",
    "T9": "Son las entidades de la tabla de la tarea 2 que cumplen DOS condiciones a la vez: mas de 5 billones de apropiacion y ejecucion por debajo de ejecucion_nacional. Se combinan con & y cada condicion entre parentesis, como en la clase 2.",
    "T10": "El conteo sale de las llamadas a guardar(...). Si esta en cero, todavia no guardo ninguna figura; si van menos de cinco, revise cual figura le falta. Los nombres tienen que ser exactamente los cinco del README."
}

_ESPERADO = {
    "T1": "a8c5aa5b2b",
    "T2": "7d17786744",
    "T6": "f9da139fa1",
    "T9": "f2473a1cb2",
    "T10": "e681962705"
}


def _firma(valor):
    """Reduce un resultado a un texto reproducible, sin importar como se calculo."""
    if isinstance(valor, pd.DataFrame):
        partes = ["DataFrame", str(valor.shape), str([str(c) for c in valor.columns]),
                  str([str(i) for i in valor.index])]
        for columna in valor.columns:
            serie = valor[columna]
            if pd.api.types.is_bool_dtype(serie) or not pd.api.types.is_numeric_dtype(serie):
                partes.append(f"{columna}:{[str(v) for v in serie.tolist()]}")
            else:
                partes.append(f"{columna}:{round(float(serie.sum()), 4)}")
        return "|".join(partes)
    if isinstance(valor, pd.Series):
        return "|".join(["Series", str(len(valor)), str([str(i) for i in valor.index]),
                         str([str(v) for v in valor.tolist()])])
    if isinstance(valor, (list, tuple)):
        return "lista|" + "|".join(str(v) for v in valor)
    if not isinstance(valor, str):
        try:
            return f"numero|{round(float(valor), 4)}"
        except (TypeError, ValueError):
            pass
    return f"otro|{valor!r}"


def _huella(valor):
    return hashlib.sha256(_firma(valor).encode("utf-8")).hexdigest()[:10]


def _redondear(valor, decimales):
    if decimales is None or valor is None:
        return valor
    if isinstance(valor, (pd.DataFrame, pd.Series)):
        return valor.round(decimales)
    try:
        return round(float(valor), decimales)
    except (TypeError, ValueError):
        return valor


def comprobar(clave, valor, decimales=None):
    """Dice si el resultado es el correcto, sin revelar cual era."""
    _RESULTADOS[clave] = False
    if valor is None:
        print(f"[{clave}] Sin resolver todavia: la variable sigue valiendo None.")
        return
    valor = _redondear(valor, decimales)
    if isinstance(valor, pd.DataFrame):
        print(f"[{clave}] Usted produjo un DataFrame de {valor.shape[0]} filas "
              f"y {valor.shape[1]} columnas.")
    elif isinstance(valor, pd.Series):
        print(f"[{clave}] Usted produjo una Series de {len(valor)} elementos.")
    else:
        print(f"[{clave}] Usted produjo: {valor!r}")
    if _huella(valor) == _ESPERADO.get(clave):
        _RESULTADOS[clave] = True
        print(f"[{clave}] CORRECTO.")
    else:
        print(f"[{clave}] Todavia no coincide.")
        print(f"[{clave}] Pista: {_PISTAS[clave]}")


def _lista_de_ejes(eje):
    if eje is None:
        return []
    if hasattr(eje, "flatten"):          # arreglo de plt.subplots(2, 4)
        return list(eje.flatten())
    if isinstance(eje, (list, tuple)):
        return list(eje)
    return [eje]


def _titulo(ax):
    """El titulo de un eje, este puesto al centro, a la izquierda o a la derecha."""
    return any(ax.get_title(loc=lado).strip() for lado in ("center", "left", "right"))


def comprobar_grafico(clave, eje, con_ejes=True, desde_cero=None, minimo_textos=0,
                      con_referencia=False, escalas_compartidas=False):
    """Revisa que el grafico exista y cumpla lo que se califica.

    No hay una unica respuesta correcta para un grafico: lo que se comprueba es
    lo que pide la checklist de diez puntos. Dibujado, titulado, etiquetado, con
    el eje en cero cuando la regla lo exige, con la referencia cuando la figura
    la necesita, y con la misma escala en todos los paneles cuando son small
    multiples.
    """
    _RESULTADOS[clave] = False
    ejes = _lista_de_ejes(eje)
    if not ejes:
        print(f"[{clave}] Sin resolver todavia: la variable del eje sigue valiendo None.")
        print(f"[{clave}] Pista: {_PISTAS[clave]}")
        return
    if not all(hasattr(e, "get_title") for e in ejes):
        print(f"[{clave}] Eso no es un eje de matplotlib.")
        print(f"[{clave}] Pista: {_PISTAS[clave]}")
        return

    faltas = []
    varios = len(ejes) > 1
    for numero, actual in enumerate(ejes, start=1):
        etiqueta = f"panel {numero}: " if varios else ""
        if not _titulo(actual):
            faltas.append(f"{etiqueta}falta el titulo: ax.set_title('...')")
        if con_ejes:
            if not actual.get_xlabel().strip():
                faltas.append(f"{etiqueta}falta la etiqueta del eje x, con su unidad")
            if not actual.get_ylabel().strip():
                faltas.append(f"{etiqueta}falta la etiqueta del eje y, con su unidad")
        dibujado = (len(actual.collections) + len(actual.lines)
                    + len(actual.patches) + len(actual.images))
        if dibujado == 0:
            faltas.append(f"{etiqueta}el eje esta vacio: el grafico no se dibujo sobre este eje")

    principal = ejes[0]
    if desde_cero in ("x", "y"):
        limites = principal.get_xlim() if desde_cero == "x" else principal.get_ylim()
        if round(min(limites), 6) != 0:
            faltas.append(f"el eje {desde_cero} no arranca en cero: en barras eso hace que la "
                          f"longitud mienta. ax.set_{desde_cero}lim(0, ...)")
    if minimo_textos and sum(len(e.texts) for e in ejes) < minimo_textos:
        faltas.append(f"faltan textos sobre el grafico: se esperan al menos {minimo_textos} "
                      f"(etiquetas directas o anotaciones con ax.text / ax.annotate)")
    if con_referencia and len(principal.lines) == 0:
        faltas.append("falta la linea de referencia: ax.axhline(...) o ax.axvline(...)")
    if escalas_compartidas:
        escalas = {tuple(round(v, 6) for v in e.get_xlim()) for e in ejes}
        if len(escalas) > 1:
            faltas.append("los paneles no comparten la escala del eje x: la comparacion visual "
                          "es falsa. plt.subplots(..., sharex=True)")

    if faltas:
        print(f"[{clave}] Todavia no esta completo:")
        for falta in faltas:
            print(f"[{clave}]   - {falta}")
        print(f"[{clave}] Pista: {_PISTAS[clave]}")
    else:
        _RESULTADOS[clave] = True
        print(f"[{clave}] CORRECTO: el grafico cumple lo que se revisa aqui. "
              f"Los cinco segundos y el titulo verdadero los juzga usted.")


def resumen_puntos_de_control():
    """Estado de las diez tareas."""
    print("Punto de control")
    print("-" * 42)
    for clave in _CLAVES:
        estado = "correcto" if _RESULTADOS.get(clave) else "pendiente"
        print(f"  {clave}: {estado}")
    logrados = sum(1 for c in _CLAVES if _RESULTADOS.get(c))
    print("-" * 42)
    print(f"{logrados} de {len(_CLAVES)} {'correctas' if logrados != 1 else 'correcta'}.")


print("Verificador listo. Las tareas se comprueban con comprobar('T1', su_variable).")

---

## Paso 0.1 · Su paleta

**Requisito no negociable: una paleta, definida aquí, usada en las cinco figuras.** Es lo primero que
se mira al calificar, porque es lo que separa cinco figuras sueltas de un informe.

**El concepto.** Una paleta es un acuerdo, no una decoración. Cuatro roles bastan: un **neutro** para
todo lo que no es el mensaje, un **énfasis** para el único elemento que importa, una **alerta**
reservada a los valores bajo umbral, y un color de **referencia** para líneas de meta y anotaciones.
El quinto, **secundario**, aparece solo cuando de verdad hay dos series (figuras 2 y 4).

**Las reglas que su paleta tiene que cumplir:**

- El neutro es grisáceo y no compite con nada.
- El énfasis tiene contraste fuerte contra el neutro, también en escala de grises.
- **Nunca rojo contra verde** como única distinción: es la peor combinación para daltonismo.
- El mismo rol tiene el mismo color en las cinco figuras. Si Educación es azul en la figura 1, es
  azul en las cinco.

Puede usar la del demo o definir la suya. Paletas de arranque, si no quiere decidir:

- Azul institucional: `'#B8C4CE'`, `'#1F4E79'`, `'#C0392B'`, `'#7F8C8D'`, `'#E67E22'`
- Tierra sobria: `'#CBC5B9'`, `'#5D4E37'`, `'#B22222'`, `'#A9A9A9'`, `'#CD853F'`
- Mínima moderna: `'#D8DEE9'`, `'#2E4057'`, `'#D64550'`, `'#8D99AE'`, `'#048A81'`

**Lo que decide usted.** Los cinco colores. Los nombres de las claves no se cambian: el resto del
cuaderno los usa.

In [ ]:
# TU CÓDIGO AQUÍ
# Rellene los cinco colores. Las claves ya están: no las renombre.

PALETA = {
    'neutro': '',       # gris de fondo, para todo lo que no es el mensaje
    'enfasis': '',      # el color del mensaje
    'alerta': '',       # solo bajo umbral crítico
    'referencia': '',   # líneas de meta y anotaciones
    'secundario': '',   # segunda serie, cuando hacen falta dos
}

FUENTE = 'Fuente: datos.gov.co - Ejecución del Presupuesto General de la Nación'

**Tu respuesta.** Justifique su paleta en dos líneas: ¿por qué esos colores y no otros? Diga
explícitamente qué pasa con ella en escala de grises.

*Tu respuesta:*

## Paso 0.2 · Las dos funciones de apoyo

Están completas. Léalas antes de usarlas: las va a llamar cinco veces cada una.

- **`limpiar_ejes`** es el data-ink ratio en código: borra los bordes del recuadro, manda la grilla
  detrás de los datos y la deja tenue y en un solo eje.
- **`guardar`** exporta a PNG *publication-ready*: `dpi=300` para que no se pixele impreso,
  `bbox_inches='tight'` para que no se corten las etiquetas y `facecolor='white'` para que no salga
  con fondo transparente al pegarla en las diapositivas. Además lleva la cuenta de lo que exportó,
  que es lo que comprueba la tarea 10.

In [ ]:
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = False

EXPORTADAS = []


def limpiar_ejes(ax, eje_grilla='x'):
    """Quita los bordes sobrantes y deja una grilla tenue en un solo eje."""
    for lado in ['top', 'right', 'left']:
        ax.spines[lado].set_visible(False)
    if eje_grilla == 'x':
        ax.xaxis.grid(True, linestyle='--', alpha=0.3)
        ax.yaxis.grid(False)
    else:
        ax.yaxis.grid(True, linestyle='--', alpha=0.3)
        ax.xaxis.grid(False)
    ax.set_axisbelow(True)
    return ax


def guardar(fig, nombre):
    """Exporta la figura a PNG publication-ready y anota que quedo exportada."""
    ruta = f'figuras/{nombre}.png'
    fig.savefig(ruta, dpi=300, bbox_inches='tight', facecolor='white')
    if nombre not in EXPORTADAS:
        EXPORTADAS.append(nombre)
    print('Guardada:', ruta)


print('Funciones listas.')

---

# Parte A · Los datos que va a graficar

**Qué se practica aquí.** Nada nuevo: es el `groupby` de la clase 4 sobre otro archivo. Está separado
en dos tareas porque **de estas dos tablas salen las cinco figuras**, y una tabla mal armada produce
cinco figuras mal armadas sin lanzar un solo error.

**Las dos columnas derivadas, y por qué importan:**

- **`% Ejecución`** es un cociente: `Compromisos / Apropiación Vigente * 100`. No depende del tamaño
  del sector, así que se puede comparar entre sectores grandes y chicos.
- **`Apropiación (billones)`** es la misma cifra dividida entre `1e12`. Es lo que va a los ejes. Sin
  esto el eje dice `1e13` y nadie entiende nada. **Es el atasco número uno del reto.**

### Tarea 1 · La tabla por sector

**La pregunta.** ¿Cuánta plata, y cuánta ejecución, tiene cada uno de los 32 sectores?

**El concepto.** `groupby('columna')[lista].sum()` separa por categoría y suma cada montón: la
analogía de los M&Ms de la clase 4. Lo que se suma son las cuatro columnas de plata. Lo que **no** se
suma nunca es un porcentaje: el `% Ejecución` se calcula **después** de agregar, sobre las sumas.
Promediar porcentajes de filas es un error clásico y silencioso.

**Los comandos.**

```python
df.groupby('columna')[['a', 'b']].sum()
tabla['nueva'] = tabla['a'] / tabla['b'] * 100
tabla['otra'] = tabla['a'] / 1e12
tabla.sort_values('columna', ascending=False)
```

**Lo que decide usted.** Sobre qué DataFrame se agrupa (¿el original o el limpio?), y el orden de la
tabla. Los nombres de las columnas derivadas sí están fijados, porque el resto del cuaderno los usa:
`'% Ejecución'` y `'Apropiación (billones)'`, con tilde y con mayúscula.

In [ ]:
# TU CÓDIGO AQUÍ
# 1. Agrupe por 'Nombre Sector' y sume las cuatro columnas de plata:
#    'Apropiación Vigente', 'Compromisos', 'Obligaciones', 'Pagos'.
# 2. Agregue '% Ejecución' = Compromisos / Apropiación Vigente * 100.
# 3. Agregue 'Apropiación (billones)' y 'Compromisos (billones)', dividiendo entre 1e12.
# 4. Ordene de mayor a menor apropiación y guarde la tabla en sector.

sector = None

In [ ]:
# La comprobación mira solo las dos columnas que importan, para que no dependa
# del orden en que usted haya creado las demás.
if sector is None:
    comprobar('T1', None)
else:
    comprobar('T1', sector[['Apropiación (billones)', '% Ejecución']], decimales=2)

### Tarea 2 · La tabla por entidad

**La pregunta.** La misma, un nivel más abajo: ¿cuánta plata y cuánta ejecución tiene cada una de las
221 entidades?

**El concepto.** Idéntico, cambiando la columna por la que se agrupa. Lo que cambia es la **densidad**
de la tabla: 221 filas en vez de 32, y eso decide el gráfico. Con 32 categorías se pueden hacer
barras; con 221 no, y por eso la figura 3 es una dispersión. **El número de categorías es un dato de
diseño, no un detalle.**

Aquí solo hacen falta dos columnas de plata: apropiación y compromisos.

**Lo que decide usted.** La columna de agrupación y el orden.

In [ ]:
# TU CÓDIGO AQUÍ
# 1. Agrupe por 'Nombre Entidad' y sume 'Apropiación Vigente' y 'Compromisos'.
# 2. Agregue '% Ejecución' y 'Apropiación (billones)', con esos nombres exactos.
# 3. Ordene de mayor a menor apropiación y guarde la tabla en entidad.

entidad = None

In [ ]:
if entidad is None:
    comprobar('T2', None)
else:
    comprobar('T2', entidad[['Apropiación (billones)', '% Ejecución']], decimales=2)

---

# Parte B · Las cinco figuras

**Qué se practica aquí.** Elegir y ejecutar el gráfico que corresponde a cada pregunta. Cada figura
responde una pregunta que ninguna otra responde: esa es la regla que decide cuántas figuras lleva un
informe.

| # | La pregunta | El gráfico | ¿Estuvo en el demo? |
|---|-------------|-----------|---------------------|
| 1 | ¿Qué sectores concentran el presupuesto? | Barras horizontales, top 10 | Sí. El calentamiento |
| 2 | ¿Dónde es más grande la brecha entre lo autorizado y lo comprometido? | Barras agrupadas, top 8 | No |
| 3 | ¿Qué entidades tienen mucho presupuesto y poca ejecución? | Dispersión con línea de referencia | No |
| 4 | ¿Cómo cambia la composición del gasto entre sectores? | Barras apiladas al 100% | No |
| 5 | ¿Qué sectores están por debajo del promedio de ejecución? | Small multiples | No |

**Lo que se exige en las cinco, y no se repite en cada tarea:**

1. Su paleta, la misma en todas.
2. Ejes con unidad legible: billones de pesos o porcentaje, nunca `1e13`.
3. Título que enuncia una conclusión, **y que es verdadera**. Verifíquelo contra la tabla.
4. La fuente citada, con `fig.text(...)`.
5. `limpiar_ejes(...)` y nada de basura visual.
6. Exportación con `guardar(fig, 'nombre')`, con el nombre exacto que dice cada tarea.

### Tarea 3 · Figura 1 · Qué sectores concentran el presupuesto

**La pregunta.** ¿Qué sectores concentran el presupuesto?

**El concepto.** Comparar magnitudes entre categorías: **barras**. Horizontales, porque los nombres de
sector son largos y en vertical se rotan hasta volverse ilegibles; la solución al texto ilegible
nunca es rotarlo más. Ordenadas, porque ordenar es lo que convierte una lista en un ranking. Y con
todo en gris salvo la primera, porque el color se reserva para el mensaje.

**Los comandos.**

```python
datos = tabla.head(10).sort_values('columna')     # ascendente: barh dibuja de abajo hacia arriba
fig, ax = plt.subplots(figsize=(10, 6))
colores = [PALETA['neutro']] * len(datos)
colores[-1] = PALETA['enfasis']
ax.barh(datos.index, datos['columna'], color=colores)
ax.text(valor + holgura, i, f'{valor:,.1f}', va='center')
ax.set_xlim(0, datos['columna'].max() * 1.15)
ax.set_xlabel('...')
ax.set_ylabel('...')
ax.set_title('...', loc='left', pad=15)
limpiar_ejes(ax, eje_grilla='x')
fig.text(0.01, -0.02, FUENTE, fontsize=8, color=PALETA['referencia'])
```

**Lo que decide usted.** El título, que tiene que decir una conclusión verdadera sobre esta tabla, y
la unidad que va en la etiqueta del eje. Guarde la figura en `fig1` y el eje en `eje1`.

**Nombre de archivo:** `fig1_apropiacion_por_sector`

In [ ]:
# TU CÓDIGO AQUÍ
# 1. datos = los 10 sectores con más apropiación, ordenados ascendente.
# 2. fig1, eje1 = plt.subplots(...)
# 3. Lista de colores: neutro para todas, énfasis para la última.
# 4. eje1.barh(...) y el bucle de etiquetas directas con eje1.text().
# 5. set_xlim desde CERO, las dos etiquetas de eje y el título con la conclusión.
# 6. limpiar_ejes(eje1) y la fuente con fig1.text(...).
# 7. guardar(fig1, 'fig1_apropiacion_por_sector')

fig1 = None
eje1 = None

In [ ]:
comprobar_grafico('T3', eje1, desde_cero='x', minimo_textos=5)

**Tu respuesta.** Escriba aquí el título que le puso y verifique que sea **verdadero** contra la
tabla de la tarea 1. Un título que afirma algo que los datos no sostienen es el error más grave que
se puede cometer hoy.

*Tu respuesta:*

### Tarea 4 · Figura 2 · Dónde es más grande la brecha

**La pregunta.** ¿En qué sectores es más grande la brecha entre lo autorizado y lo comprometido?

**El concepto.** Dos series por categoría: **barras agrupadas**. matplotlib no tiene una función para
esto; se hace desplazando las posiciones a mano. La idea es simple: en vez de dibujar una barra en la
posición 0, se dibujan dos, una un poco arriba y otra un poco abajo del centro.

```python
posiciones = np.arange(len(datos))   # 0, 1, 2, ...
ancho = 0.38
ax.barh(posiciones - ancho/2, serie_1, height=ancho, label='...')
ax.barh(posiciones + ancho/2, serie_2, height=ancho, label='...')
ax.set_yticks(posiciones)
ax.set_yticklabels(datos.index)
ax.legend(frameon=False)
```

**La decisión de diseño que sí tiene que tomar.** Aquí **sí** hacen falta dos colores, porque son dos
series distintas: ya no aplica el gris más resaltado tal cual. Use `enfasis` y `secundario`, y
verifique que se distingan **en escala de grises**: azul contra naranja funciona, azul contra azul
claro no.

**Y el número que hace legible la figura:** etiquete el `% Ejecución` al final de cada par. Sin ese
número el lector ve el tamaño de la brecha, pero no sabe si un sector con brecha enorme está mal o si
es simplemente gigante. La brecha absoluta y la brecha porcentual no son el mismo ranking.

**Lo que decide usted.** El recorte (top 8), qué serie va arriba y cuál abajo, y el título.
Guarde la figura en `fig2` y el eje en `eje2`.

**Nombre de archivo:** `fig2_brecha_apropiacion_compromisos`

In [ ]:
# Esta celda usa la tabla de la tarea 1. Si sector sigue en None, resuélvala primero.
datos_f2 = None

if sector is None:
    print('Pendiente: resuelva la tarea 1 y vuelva a ejecutar esta celda.')
else:
    datos_f2 = sector.head(8).sort_values('Apropiación (billones)')
    datos_f2 = datos_f2.assign(**{'Brecha (billones)': datos_f2['Apropiación (billones)'] -
                                                       datos_f2['Compromisos (billones)']})
    print(datos_f2[['Apropiación (billones)', 'Compromisos (billones)',
                    'Brecha (billones)', '% Ejecución']].round(2).to_string())

In [ ]:
# TU CÓDIGO AQUÍ
# 1. fig2, eje2 = plt.subplots(figsize=(11, 7))
# 2. posiciones = np.arange(len(datos_f2)) y ancho = 0.38
# 3. Dos barh desplazados, con height=ancho, color y label distintos.
# 4. set_yticks / set_yticklabels con los nombres de sector.
# 5. Etiquete el % de ejecución al final de cada par con eje2.text().
# 6. Eje x desde CERO, las dos etiquetas de eje, título con la conclusión.
# 7. legend(frameon=False), limpiar_ejes(eje2), fuente al pie.
# 8. guardar(fig2, 'fig2_brecha_apropiacion_compromisos')

fig2 = None
eje2 = None

In [ ]:
comprobar_grafico('T4', eje2, desde_cero='x', minimo_textos=4)

**Tu respuesta.** ¿En qué sector es más grande la brecha en valor absoluto? ¿Y en porcentaje? ¿Son
el mismo sector? Esa diferencia es exactamente la razón por la que la figura lleva el porcentaje
etiquetado.

*Tu respuesta:*

### Tarea 5 · Figura 3 · Mucho presupuesto y poca ejecución

**La pregunta.** ¿Qué entidades tienen mucho presupuesto y poca ejecución?

**El concepto.** Relación entre dos variables numéricas, con 221 observaciones: **dispersión**
(*scatter*), que es la tabla de elección de gráfico de la clase 5. Ningún otro gráfico responde esta
pregunta igual de bien, porque la pregunta es sobre un **cuadrante**: abajo a la derecha, mucha plata
y poca ejecución.

**El problema real que va a encontrar.** La apropiación va desde miles de millones hasta 97 billones.
En escala lineal, las 200 entidades pequeñas se apelotonan contra el eje izquierdo y no se ve nada.
La salida es la **escala logarítmica** en el eje x, y aquí hay una decisión honesta que tomar: la
escala logarítmica **comprime las diferencias grandes**. Es el mismo tradeoff del eje truncado en otra
forma, y la regla es la misma: **si la usa, la declara** en la etiqueta del eje.

**La línea de referencia no es opcional.** Un 8% de ejecución no significa nada suelto. Contra el
promedio nacional, significa "la mitad del país". Es tinta que no representa datos y es lo más útil
de la figura.

**Los comandos.**

```python
ax.scatter(x, y, color=..., alpha=0.55, s=40)
ax.set_xscale('log')
ax.axhline(ejecucion_nacional, color=..., linestyle='--')
ax.annotate('texto', xy=(x, y), xytext=(x2, y2),
            arrowprops=dict(arrowstyle='->', color=...))
```

**Lo que decide usted.** Qué dos entidades vale la pena anotar (**al menos dos**), y por qué esas. La
celda de abajo le muestra las candidatas del cuadrante crítico. Guarde la figura en `fig3` y el eje
en `eje3`.

**Nombre de archivo:** `fig3_entidades_presupuesto_vs_ejecucion`

In [ ]:
# Esta celda usa la tabla de la tarea 2. Si entidad sigue en None, resuélvala primero.
candidatas = None

if entidad is None:
    print('Pendiente: resuelva la tarea 2 y vuelva a ejecutar esta celda.')
else:
    candidatas = entidad[(entidad['Apropiación (billones)'] > 5) &
                         (entidad['% Ejecución'] < ejecucion_nacional)]
    candidatas = candidatas.sort_values('Apropiación (billones)', ascending=False)
    print(f'Referencia nacional: {ejecucion_nacional:.1f}%')
    print(f'Entidades en el cuadrante crítico: {len(candidatas)}')
    print(candidatas[['Apropiación (billones)', '% Ejecución']].round(2).head(6).to_string())

In [ ]:
# TU CÓDIGO AQUÍ
# 1. fig3, eje3 = plt.subplots(figsize=(11, 7))
# 2. eje3.scatter(...) con las 221 entidades, alpha bajo para ver la densidad.
# 3. eje3.set_xscale('log') y DECLÁRELO en la etiqueta del eje x.
# 4. eje3.axhline(ejecucion_nacional, linestyle='--') con su etiqueta.
# 5. Al menos DOS entidades anotadas con eje3.annotate(...).
# 6. Las dos etiquetas de eje y el título con la conclusión.
# 7. limpiar_ejes(eje3, eje_grilla='y'), fuente al pie.
# 8. guardar(fig3, 'fig3_entidades_presupuesto_vs_ejecucion')

fig3 = None
eje3 = None

In [ ]:
comprobar_grafico('T5', eje3, con_referencia=True, minimo_textos=2)

**Tu respuesta.** Nombre dos entidades del cuadrante "mucho presupuesto, poca ejecución" y diga por
qué cree que están ahí. **Márquelo como hipótesis, no como conclusión:** con estos datos usted sabe
*qué* pasa, no *por qué*. Es la misma disciplina de la clase 5: correlación no es causalidad, y aquí
ni siquiera hay correlación, hay una foto.

*Tu respuesta:*

### Tarea 6 · La tabla de composición, normalizada

**La pregunta.** De cada peso que tiene un sector, ¿qué proporción va a funcionamiento, cuánta a
inversión y cuánta al servicio de la deuda?

**El concepto.** Esta tarea es la **única línea de pandas de verdad** del reto, y es la que más se
atasca. El `pivot_table` de abajo ya está escrito: cruza sector con categoría de rubro y suma la
apropiación de cada celda. Lo que falta es **normalizar por fila**: dividir cada fila por su propio
total para que las tres categorías sumen 100.

```python
porcentajes = tabla.div(tabla.sum(axis=1), axis=0) * 100
```

Los dos `axis` dicen cosas distintas y por eso confunden:

- `tabla.sum(axis=1)` — **sume a lo ancho**: un total por fila.
- `tabla.div(..., axis=0)` — **divida alineando por filas**: a cada fila su propio total.

Si le sale que cada fila suma 300, o valores gigantes, invirtió los `axis`.

**Si esto le toma más de cinco minutos, copie la línea y siga.** Lo que se evalúa hoy es el diseño, no
el pivoteo.

**Lo que decide usted.** Nada más que aplicar la línea sobre la tabla correcta. Guarde el resultado en
`composicion_pct`.

In [ ]:
# Esta celda usa la tabla de la tarea 1. Si sector sigue en None, resuélvala primero.
composicion = None

if sector is None:
    print('Pendiente: resuelva la tarea 1 y vuelva a ejecutar esta celda.')
else:
    composicion = df_limpio.pivot_table(
        index='Nombre Sector',
        columns='Nombre Nivel Uno Rubro',
        values='Apropiación Vigente',
        aggfunc='sum',
        fill_value=0,
    ).loc[sector.head(8).index]
    print(composicion.round(0).to_string())

In [ ]:
# TU CÓDIGO AQUÍ
# 1. Divida cada fila de composicion por su propio total y multiplique por 100.
# 2. Guarde el resultado en composicion_pct.
# 3. Verifique: imprima la suma por fila. Todas tienen que dar 100.

composicion_pct = None

In [ ]:
if composicion_pct is None:
    comprobar('T6', None)
else:
    comprobar('T6', composicion_pct.sort_index()[sorted(composicion_pct.columns)], decimales=2)

### Tarea 7 · Figura 4 · La composición del gasto

**La pregunta.** ¿Cómo cambia la composición del gasto entre sectores?

**El concepto.** Proporciones de un todo, comparadas entre categorías: **barras apiladas al 100%**.
Es la respuesta correcta a "¿qué proporción del total?" cuando hay varias categorías que comparar, y
es la razón por la que casi nunca hace falta un pie: aquí habría que dibujar ocho pies y compararlos
de a pares, que es exactamente lo que el ojo no sabe hacer.

**Cómo se apila.** matplotlib no apila solo: hay que llevar un acumulado y decirle a cada capa dónde
empieza.

```python
acumulado = np.zeros(len(datos))
for categoria in datos.columns:
    ax.barh(datos.index, datos[categoria], left=acumulado, label=categoria, color=...)
    acumulado += datos[categoria].values
```

**Sobre el color aquí.** Tres categorías que tienen un orden conceptual (de gasto corriente a deuda)
piden una **escala secuencial**: tres tonos del mismo color, del claro al oscuro, en vez de tres
colores sueltos. Un ejemplo válido con su paleta: neutro, secundario, énfasis. Lo que **no** funciona
es rojo, verde y amarillo.

**Y la lectura:** etiquete el porcentaje dentro de cada segmento solo cuando quepa, digamos por
encima del 8%. Un número que no cabe es basura visual.

**Lo que decide usted.** Por qué categoría ordena las barras, los tres tonos y el título. El eje x va
de 0 a 100. Guarde la figura en `fig4` y el eje en `eje4`.

**Nombre de archivo:** `fig4_composicion_gasto_por_sector`

In [ ]:
# TU CÓDIGO AQUÍ
# 1. datos = composicion_pct ordenada por la categoría que quiera destacar.
# 2. fig4, eje4 = plt.subplots(figsize=(11, 6))
# 3. Bucle de capas con left=acumulado, y acumulado += valores de la capa.
# 4. Etiquetas de porcentaje dentro de los segmentos mayores a 8.
# 5. eje4.set_xlim(0, 100), las dos etiquetas de eje, título con la conclusión.
# 6. legend arriba y sin caja, limpiar_ejes(eje4), fuente al pie.
# 7. guardar(fig4, 'fig4_composicion_gasto_por_sector')

fig4 = None
eje4 = None

In [ ]:
comprobar_grafico('T7', eje4, desde_cero='x')

**Tu respuesta.** ¿Qué sector tiene la composición más distinta de todos los demás? ¿Qué le dice eso
sobre su función dentro del Estado?

*Tu respuesta:*

### Tarea 8 · Figura 5 · Small multiples

**La pregunta.** ¿Qué sectores están por debajo del promedio de ejecución?

**El concepto.** Los **small multiples** son la misma figura repetida, una por grupo, en una rejilla.
Son la respuesta correcta cuando hay que comparar la misma medida entre muchos grupos sin encimarlos
en una sola figura: el ojo compara paneles vecinos muy bien, siempre que los paneles sean idénticos
salvo por los datos.

**La regla que decide si la figura sirve o miente: `sharex=True`.** Sin escalas compartidas, cada
panel se autoescala y un 3% ocupa el mismo ancho que un 45%. La comparación visual queda invertida y
la figura **se ve perfectamente bien**. Es exactamente el tema de la clase, en su forma más
traicionera, y por eso el verificador lo revisa.

**El color, respaldado.** Los sectores por debajo del promedio nacional van en su color de alerta.
Y el color no puede ser el único canal: la línea de referencia y la etiqueta del valor dicen lo mismo,
así que quien no distinga los colores recibe el mensaje igual.

**Los comandos.**

```python
fig, ejes = plt.subplots(2, 4, figsize=(15, 7), sharex=True)
for ax, (nombre, fila) in zip(ejes.flatten(), datos.iterrows()):
    ax.barh([0], [fila['% Ejecución']], height=0.5, color=...)
    ax.axvline(ejecucion_nacional, linestyle='--')
    ax.set_title(nombre, fontsize=9)
    ax.set_yticks([])
fig.suptitle('...')
```

**Lo que decide usted.** El orden de los paneles, el título general y qué se etiqueta dentro de cada
uno. **Guarde el arreglo completo de ejes en `ejes5`**, no un panel suelto: la comprobación revisa los
ocho. La figura va en `fig5`.

**Nombre de archivo:** `fig5_ejecucion_small_multiples`

In [ ]:
# TU CÓDIGO AQUÍ
# 1. datos = los 8 sectores más grandes, ordenados por % de ejecución.
# 2. fig5, ejes5 = plt.subplots(2, 4, figsize=(15, 7), sharex=True)   <- sharex NO es opcional
# 3. Recorra ejes5.flatten() junto con las filas de datos.
# 4. En cada panel: barh del valor, axvline de la referencia, título con el nombre
#    del sector, etiqueta del valor, sin ticks en y.
# 5. Un solo título general con fig5.suptitle() y una sola fuente al pie.
# 6. guardar(fig5, 'fig5_ejecucion_small_multiples')

fig5 = None
ejes5 = None

In [ ]:
comprobar_grafico('T8', ejes5, con_ejes=False, escalas_compartidas=True, minimo_textos=8)

**Tu respuesta.** ¿Cuántos de los 8 sectores están por debajo del promedio nacional? Cuéntelos en la
figura y verifíquelo contra la tabla.

*Tu respuesta:*

**Tu respuesta.** Quite `sharex=True`, vuelva a ejecutar y mire. ¿La figura se ve mal, o se ve bien y
engaña? Diga qué conclusión sacaría alguien que la mirara cinco segundos.

*Tu respuesta:*

---

# Parte C · El ensamblaje

**Es la parte que más pesa, y es la única donde no se le da el orden de los comandos.**

Todo lo que necesita ya lo usó hoy o en las clases 2 y 4. **Lo que no se le da es la secuencia**, y
eso es deliberado: en las sustentaciones de los momentos 1, 2 y 3 nadie le va a dar el orden.

### El inventario de comandos

```python
tabla[(tabla['a'] > valor) & (tabla['b'] < otro)]
tabla.sort_values('columna', ascending=False)
tabla[['columna_a', 'columna_b']]
tabla.round(2)
len(tabla)
guardar(figura, 'nombre_del_archivo')
```

Nada más. Si está buscando una función nueva, se pasó de largo: las dos tareas de esta parte se
resuelven con lo de arriba.

### Tarea 9 · Las entidades críticas

**La pregunta.** ¿Cuáles son las entidades grandes que están ejecutando por debajo del promedio
nacional?

**El concepto.** Es la pregunta de la figura 3 convertida en una tabla, y la tabla es lo que se cita
en una sustentación: un gráfico muestra el patrón, pero cuando alguien pregunta "¿cuáles
exactamente?", se necesita la lista. **Dos condiciones a la vez**, combinadas con `&` y cada una
entre paréntesis, como en la clase 2:

- Apropiación por encima de **5 billones** de pesos. Es el filtro de tamaño: sin él la lista se llena
  de entidades diminutas cuyo porcentaje se mueve con nada.
- Ejecución por debajo de `ejecucion_nacional`.

**El umbral de 5 billones es una decisión**, no un dato. Si lo cambia, cambia la lista, y tiene que
poder defenderlo. Aquí se fija en 5 para que la tabla sea del tamaño de una diapositiva.

**Lo que decide usted.** El ensamblaje completo: qué tabla, en qué orden se aplican las cosas, con qué
columnas se queda y cómo la ordena. Guarde el resultado en `entidades_criticas`.

In [ ]:
# TU CÓDIGO AQUÍ
# 1. De la tabla por entidad, quédese con las que tienen más de 5 billones de
#    apropiación Y menos ejecución que ejecucion_nacional.
# 2. Deje solo las columnas 'Apropiación (billones)' y '% Ejecución'.
# 3. Ordene de mayor a menor apropiación y guarde en entidades_criticas.
# 4. Imprima cuántas son y muestre la tabla.

entidades_criticas = None

In [ ]:
if entidades_criticas is None:
    comprobar('T9', None)
else:
    comprobar('T9', entidades_criticas[['Apropiación (billones)', '% Ejecución']].sort_index(),
              decimales=2)

**Tu respuesta.** Si tuviera que llevar esta tabla a una reunión, ¿la mostraría completa o solo las
tres primeras filas? Justifique con la pregunta que quiere que responda quien la mire.

*Tu respuesta:*

### Tarea 10 · La exportación

**La pregunta.** ¿Están las cinco figuras exportadas, con los nombres correctos y en calidad de
publicación?

**El concepto.** Una figura que solo existe dentro del notebook no se puede pegar en una diapositiva
ni en un informe. La exportación tiene tres parámetros que no son negociables, y `guardar()` ya los
trae:

| Parámetro | Qué pasa sin él |
|-----------|-----------------|
| `dpi=300` | El PNG sale borroso al proyectarlo o imprimirlo |
| `bbox_inches='tight'` | Las etiquetas de los bordes salen cortadas |
| `facecolor='white'` | El fondo sale transparente y se ve sucio sobre las diapositivas |

**Los cinco nombres, exactos:**

```
fig1_apropiacion_por_sector
fig2_brecha_apropiacion_compromisos
fig3_entidades_presupuesto_vs_ejecucion
fig4_composicion_gasto_por_sector
fig5_ejecucion_small_multiples
```

**Lo que decide usted.** Nada, y por eso es la última: es la verificación de que las cinco figuras
existen de verdad. Si ya llamó a `guardar(...)` dentro de cada tarea, esta celda no tiene que hacer
nada; si no, llámela ahora para las que falten.

In [ ]:
# TU CÓDIGO AQUÍ
# Si alguna figura no quedó exportada, guárdela ahora:
#   guardar(fig1, 'fig1_apropiacion_por_sector')
#   ... y así con las cinco.

print('Figuras exportadas hasta ahora:', len(EXPORTADAS))

In [ ]:
comprobar('T10', sorted(EXPORTADAS))

---

## Punto de control

Ejecute la celda de abajo para ver cuántas de las diez tareas quedaron correctas.

Si alguna sigue pendiente, no pase de largo. Si está en el salón, levante la mano ahora, que el
profesor está aquí para eso.

In [ ]:
resumen_puntos_de_control()

---

## La checklist, figura por figura

El verificador revisó lo que una máquina puede revisar. Los tres puntos que no puede revisar nadie
más que usted son el 1, el 7 y el 8, y son los que más pesan al calificar. Márquelos a mano.

| # | Criterio | F1 | F2 | F3 | F4 | F5 |
|---|----------|----|----|----|----|----|
| 1 | Pasa la prueba de los 5 segundos con alguien de otro equipo | | | | | |
| 2 | Tipo de gráfico correcto para la pregunta | | | | | |
| 3 | Ordenado por valor | | | | | |
| 4 | Eje desde cero, o razón declarada | | | | | |
| 5 | Sin basura visual | | | | | |
| 6 | Color con propósito, y no como único canal | | | | | |
| 7 | Funciona en escala de grises | | | | | |
| 8 | Título con conclusión, y verdadera | | | | | |
| 9 | Ejes con unidades legibles | | | | | |
| 10 | Fuente citada | | | | | |

**Tu respuesta.** ¿Qué figura falla más puntos? ¿Qué le falta, concretamente?

*Tu respuesta:*

---

# Análisis crítico

Responda en español, dos o tres frases por pregunta. Estas no se comprueban con código: son las que
se leen en la dimensión **Ser**.

**1.** ¿Cuál de las cinco figuras comunica mejor el hallazgo más importante del dataset? ¿Por qué?

*Tu respuesta:*

**2.** ¿Qué se pierde si solo se muestran los sectores con más presupuesto? Piense en los 24 sectores
que quedaron fuera del recorte.

*Tu respuesta:*

**3.** Si solo pudiera mostrar una figura a un tomador de decisiones, ¿cuál, y qué pregunta responde
mejor que las otras cuatro?

*Tu respuesta:*

**4.** ¿Qué dato adicional —una columna que este archivo no tiene— haría estas figuras mucho más
útiles? Diga qué gráfico nuevo habilitaría.

*Tu respuesta:*

**5. Del reto al Momento 2.** ¿Cuál de estos cinco tipos de gráfico va a usar en el dashboard de su
equipo, y para qué pregunta? El dashboard pide tres gráficos, no diez: cada uno tiene que responder
algo que ningún otro responde.

*Tu respuesta:*

---

## Opcional · Solo si terminó todo

No se comprueban y no entran en la retroalimentación.

**A. La prueba de la fotocopia.** Vuelva a exportar una de las cinco figuras en escala de grises
(`cmap='gray'` no sirve aquí: cambie los colores de la paleta por tres grises distintos) y compruebe
si sigue funcionando. Es el punto 7 de la checklist, hecho en serio.

**B. SVG.** `fig.savefig('figuras/fig1.svg', format='svg')`. Es vectorial y no se pixela nunca. Útil
para el dashboard del Momento 2.

**C. Un archivo `estilo.py`.** Mueva su paleta, `limpiar_ejes` y `guardar` a un archivo aparte e
impórtelo. Va a producir muchos gráficos más en las clases 9, 10 y 12, y la consistencia entre
figuras es la mitad de la nota de un informe.

In [ ]:
# TU CÓDIGO AQUÍ (opcional)

---

## Antes de entregar

1. **Kernel → Restart and Run All.** Si algo revienta, arréglelo. Un cuaderno que no corre de arriba
   a abajo le pone techo a la dimensión Hacer.
2. Ejecute el punto de control y verifique que las diez tareas están correctas.
3. Verifique que **todas** las celdas *Tu respuesta:* están escritas. La figura no es el análisis.
4. Relea los cinco títulos y compruebe, uno por uno, que son **verdaderos** contra las tablas.
5. Confirme que la carpeta `figuras/` tiene los cinco PNG con los nombres pedidos.
6. Guarde como `reto_clase08_APELLIDO.ipynb` y súbalo antes del inicio de la clase 9.